# Aerial Watch: train the pig detector on BAMBI (Google Colab)

Free **T4 GPU** (Runtime → Change runtime type → T4 GPU). About 1–2 hours per model (60 epochs, ~4,000 images).

**BAMBI only (public, CC BY 4.0).** NACAS photos never go to Colab or Google Drive: they are fine-tuned on Gaurav's PC (docs/11, docs/16).

Put these in Google Drive under `MyDrive/AerialWatch/` (made by `tools/bambi_pipeline.ps1`, step 9):

| File | Content |
|---|---|
| `ml.zip` | `src/ml` code |
| `yolo-bambi.zip` | thermal dataset |
| `yolo-bambi-rgb.zip` | RGB dataset (optional, second run) |

Run all cells once with `CAMERA = "thermal"`, then (optional) again with `CAMERA = "rgb"`.

In [ ]:
CAMERA = "thermal"        # "thermal" first; then "rgb" for the second model
EPOCHS = 60
MODEL = "yolov8s.pt"       # if the GPU is unavailable/slow: "yolov8n.pt" and EPOCHS = 40
DATASET = "yolo-bambi" if CAMERA == "thermal" else "yolo-bambi-rgb"
NAME = "bambi-thermal" if CAMERA == "thermal" else "bambi-rgb"
print(CAMERA, DATASET, NAME)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/AerialWatch'

In [ ]:
# Windows "Send to > Compressed folder" (or the script) makes zips that hold "ml/..." and "yolo-bambi/..."
!rm -rf /content/ml /content/data/$DATASET && mkdir -p /content/data
!unzip -q -o $ROOT/ml.zip -d /content
!unzip -q -o $ROOT/$DATASET.zip -d /content/data
%cd /content/ml
!pip -q install -r requirements-train.txt

### Check the labels before spending GPU time
The green boxes must sit on the pigs. If not, stop and fix the conversion on the PC (offsets).

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f'/content/data/{DATASET}/check/*.jpg'))[:4]:
    display(Image(p, width=480))
!head -c 1500 /content/data/$DATASET/dataset-card.json

### Train
`--imgsz 1024` keeps small pigs visible. The data.yaml path is rewritten to where the data sits in Colab.

In [ ]:
!sed -i "s#^path: .*#path: /content/data/$DATASET#" /content/data/$DATASET/data.yaml
!python tools/train.py --data /content/data/$DATASET/data.yaml --model $MODEL --epochs $EPOCHS --imgsz 1024 --batch 8 --name $NAME --camera $CAMERA

### Evaluate on the held-out test flights: trained model vs. the no-training baseline (thermal only)

In [ ]:
import glob
weights = sorted(glob.glob(f'/content/ml/models/*-{NAME}-*.pt'))[-1]
print(weights)
!python tools/evaluate.py --data /content/data/$DATASET --split test --weights "$weights" --out /content/metrics
if CAMERA == "thermal":
    !python tools/evaluate.py --data /content/data/$DATASET --split test --baseline --out /content/metrics
!cat /content/metrics/*.md

### Save the model, its card, the metrics and the training curves to Drive
Then share `MyDrive/AerialWatch/results/` with Gaurav.

In [ ]:
!mkdir -p $ROOT/results/$NAME
!cp /content/ml/models/*-$NAME-* $ROOT/results/$NAME/
!cp /content/metrics/* $ROOT/results/$NAME/
!cp /content/ml/runs/detect/$NAME*/results.png /content/ml/runs/detect/$NAME*/confusion_matrix.png /content/ml/runs/detect/$NAME*/PR_curve.png $ROOT/results/$NAME/ 2>/dev/null; ls $ROOT/results/$NAME

### (Optional) ONNX export for a CPU-only server

In [ ]:
# from ultralytics import YOLO
# YOLO(weights).export(format='onnx', imgsz=1024)